# W14 · MotrixLab 框架：环境注册与训练工作流

> 阶段二（M5–M6）：MotrixLab —— 国产 Sim2Real 能力 · 第 2/6 讲

上一讲认识了物理引擎 MotrixSim。本讲的主角是构建在它之上的强化学习框架 **MotrixLab**：
它把机器人模型包装成可训练的 RL 环境，并接入现成的 PPO 训练框架。学完本讲，
你将完整走通「预览环境 → 训练 → 看曲线 → 部署测试」的 MotrixLab 工作流，
并读懂一个真实四足任务的环境设计。

## 学习目标

1. 画出 MotrixLab 的分层架构（`motrix_envs` / `motrix_rl`），并说出每层职责；
2. 独立完成 MotrixLab 安装，跑通 `view.py` / `train.py` / `play.py` 三件套；
3. 读懂训练命令的全部命令行参数与配置体系；
4. 以 `dm-quadruped-walk` 为例，分析一个真实机器人任务的观测/动作/奖励设计；
5. 建立「MotrixLab 工作流 ≡ 加强版 SB3 工作流」的心智映射。

> **本讲可执行性说明**：MotrixLab 命令行示例保持未执行（附预期输出描述）；
> 为建立心智模型，我们用本机已有的 SB3 + Gymnasium 真实执行一遍同构的工作流。

## 2.1 MotrixLab 架构：两层分工

MotrixLab（[GitHub: Motphys/MotrixLab](https://github.com/Motphys/MotrixLab)）的代码分为两个核心组件：

| 组件 | 职责 | 类比阶段一 |
|------|------|-----------|
| `motrix_envs` | 基于 MotrixSim 构建的 RL 环境：定义**观测、动作、奖励**。框架无关（不绑定任何训练库） | `gymnasium.Env` 及自定义 `DroneHoverEnv` |
| `motrix_rl` | 接入训练框架，读取 `motrix_envs` 的环境参数跑训练。目前支持 **SKRL**（JAX/PyTorch 后端）与 **RSLRL**（PyTorch）的 PPO | SB3 的 `PPO(...).learn()` |

这个分层和阶段一的「Gymnasium 环境 + SB3 算法」完全同构，刻意记住这个映射，
读 MotrixLab 源码时就不会迷路：

```
阶段一 (SB3)                        阶段二 (MotrixLab)
┌─────────────────────┐            ┌──────────────────────────┐
│ SB3: PPO/SAC/TD3    │            │ motrix_rl: SKRL / RSLRL  │  ← 算法层
├─────────────────────┤            ├──────────────────────────┤
│ Gymnasium Env API   │            │ motrix_envs: 环境注册     │  ← 环境层
├─────────────────────┤            ├──────────────────────────┤
│ env 内部简化动力学   │            │ MotrixSim 物理引擎        │  ← 物理层
└─────────────────────┘            └──────────────────────────┘
```

与 SB3 工作流的关键差异只有一个：**规模**。SB3 的向量化环境通常是 8–32 个 CPU 子进程；
MotrixLab 默认 `--num-envs 2048`，所有环境在同一个进程内批量推进（就是上一讲的批量 `SceneData`），
这也是它能把 PPO 采样吞吐拉高几个数量级的原因。

## 2.2 安装（详见 docs/phase2_motrixlab.md）

MotrixLab 是一个**独立项目**（自带 uv 工程），不建议装进本工程的 `.venv`，
推荐在 `~/workspace/MotrixLab` 之类的独立目录安装：

```bash
git clone https://github.com/Motphys/MotrixLab
cd MotrixLab
git lfs pull                              # 拉取模型等大文件（需先装 git-lfs）
uv sync --all-packages --all-extras       # 安装全部依赖（SKRL 双后端 + RSLRL）
```

如果只想要一个训练后端，可以精简安装（如 `--extra skrl-torch` 或 `--extra rslrl`）。
下面的三件套命令演示以官方 `cartpole` 环境为例。

> **运行前提**：MotrixLab 已在独立目录安装完成，且当前 shell 位于 MotrixLab 仓库根目录。
> 本机未安装 MotrixLab，以下 cell 保持未执行。

### 三件套之一：`view.py` —— 训练前先看环境

在烧算力训练之前，先用随机动作把环境跑起来看一眼，确认安装与模型加载都正常：

> **预期输出**：弹出渲染窗口，倒立摆小车在随机动作下左右乱晃、摆杆倒下后环境重置，
> 窗口持续运行直到手动关闭。这一步能跑通，说明 MotrixSim 物理仿真与渲染链路已就绪。

In [ ]:
# ⚠️ 运行前提：MotrixLab 已安装且当前目录为 MotrixLab 仓库根目录；本 cell 保持未执行
# 环境预览：不训练，只用随机动作演示环境（用于检测环境依赖是否配置正确）
uv run scripts/view.py --env cartpole

### 三件套之二：`train.py` —— 启动 PPO 训练

> **预期输出**：终端滚动打印训练日志（迭代次数、平均回报、耗时等），
> 训练结果保存在 `runs/cartpole/` 目录下（检查点 + TensorBoard 日志）。
> 加 `--render` 可打开可视化窗口观察学习过程（训练中按**空格键**可切换渲染开关，
> 关闭渲染可显著提速）。cartpole 这类简单任务，回报曲线应在数百次迭代内爬升到接近满分。

In [ ]:
# ⚠️ 运行前提：同上；本 cell 保持未执行

# 基础训练：默认 SKRL 框架，自动选择训练后端（JAX 或 PyTorch）
uv run scripts/train.py --env cartpole

# 指定 RL 框架为 RSLRL（仅 PyTorch 后端）
uv run scripts/train.py --env cartpole --rllib rslrl

# 高级配置：指定并行环境数 / 训练后端 / 仿真后端
uv run scripts/train.py --env cartpole \
  --rllib skrl --train-backend jax --sim-backend np --num-envs 1024

# 可视化训练（空格键切换渲染开关；渲染会显著降低训练速度，仅用于调试演示）
uv run scripts/train.py --env cartpole --render

In [ ]:
# ⚠️ 运行前提：同上；本 cell 保持未执行
# 用 TensorBoard 查看训练曲线（回报、损失、 episode 长度等标量）
uv run tensorboard --logdir runs/cartpole

### 三件套之三：`play.py` —— 测试训练好的策略

> **预期输出**：渲染窗口中倒立摆小车能长时间保持摆杆直立（策略已学会平衡）。
> 不加 `--policy` 时系统会自动在 `runs/cartpole/` 下寻找最新、最佳的策略文件。

In [ ]:
# ⚠️ 运行前提：同上且已完成一次训练；本 cell 保持未执行

# 自动寻找最佳策略测试（推荐）
uv run scripts/play.py --env cartpole

# 手动指定策略文件 + 指定测试环境数量
uv run scripts/play.py --env cartpole --policy runs/cartpole/nn/best_agent.pickle --num-envs 100

## 2.3 命令行参数与配置体系

`train.py` 支持的命令行参数（[官方文档 · 训练执行和结果分析](https://motrixlab.readthedocs.io/zh-cn/latest/user_guide/tutorial/training_and_result.html)）：

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `--env` | 环境名称 | `cartpole` |
| `--rllib` | RL 框架（skrl / rslrl） | `skrl` |
| `--sim-backend` | 仿真后端（np） | 自动选择 |
| `--train-backend` | 训练后端（jax / torch，仅 SKRL） | 自动选择 |
| `--num-envs` | 并行环境数量 | **2048** |
| `--render` | 启用渲染 | False |

注意官方文档的提醒：**学习率、网络结构等算法超参不在命令行里**，
需要通过配置文件（Python 数据类）设置——SKRL 支持为 JAX/Torch 两个后端分别配置参数，
RSLRL 使用 `RslrlCfg` 数据类。这与 SB3「超参全在构造函数里」的风格不同：
MotrixLab 沿用 legged_gym / Isaac Lab 系列的「**配置即代码**」传统，
每个环境的物理参数、奖励权重、算法超参都集中在配置对象中，便于版本管理与复现。

## 2.4 心智模型迁移：先用 SB3 走一遍同构流程

MotrixLab 装好之前，我们不妨用已有的 SB3 把「预览 → 训练 → 评估」这条线在本机真实跑一遍。
刻意对比两者的对应关系，之后接触 MotrixLab 时只需替换命令，思维框架原样复用。

> 本 cell 真实执行：PPO 在 CartPole 上训练 100k 步（CPU 约 3–5 分钟）。

In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from robot_rl_learn.common.paths import make_run_dir

# ---- 对应 view.py：先看看环境长什么样 ----
env = gym.make("CartPole-v1")
print("观测空间:", env.observation_space)   # 类比 MotrixLab 环境的 observation_space
print("动作空间:", env.action_space)

# ---- 对应 train.py：PPO 训练（MotrixLab 里也是 PPO，只是 num_envs=2048） ----
run_dir = make_run_dir("phase2_cartpole_sb3")
model = PPO("MlpPolicy", env, seed=7, verbose=0, device="cpu")
model.learn(total_timesteps=100_000)

# ---- 对应 play.py + tensorboard：评估并汇报 ----
mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"训练后评估：mean_reward = {mean_r:.1f} ± {std_r:.1f}  (满分 500)")

ckpt = run_dir / "ppo_cartpole_phase2.zip"
model.save(ckpt)
print("策略已保存到:", ckpt)

观测空间: Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
动作空间: Discrete(2)


/data/wangf/robot_rl_learn/.venv/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


训练后评估：mean_reward = 500.0 ± 0.0  (满分 500)
策略已保存到: /data/wangf/robot_rl_learn/runs/phase2_cartpole_sb3/ppo_cartpole_phase2.zip


## 2.5 解剖一个真实任务：`dm-quadruped-walk`

cartpole 只是开胃菜。MotrixLab 的 `motrix_envs` 里注册了 DeepMind Control Suite 风格的
四足任务族（[官方文档 · 四足机器人](https://motrixlab.readthedocs.io/zh-cn/latest/user_guide/demo/dm_quadruped.html)）：
`dm-quadruped-walk`（0.5 m/s 行走）、`dm-quadruped-run`（5.0 m/s 奔跑）、
`dm-quadruped-escape`（起伏地形逃离）、`dm-quadruped-fetch`（推球到目标区）。
以 walk 为例，看它比 CartPole 复杂在哪：

**动作空间**：`Box(-1, 1, (12,))` —— 4 条腿 × 每腿 3 个执行器（`yaw` 偏航 / `lift` 抬升 / `extend` 伸展），
其中 `lift`/`extend` 通过 tendon 耦合驱动髋膝踝多个关节。

**观测空间**：54 维，构成如下——

| 部分 | 内容 | 维度 |
|------|------|------|
| egocentric dof pos | 身体广义位置（本体坐标系） | 16 |
| egocentric dof vel | 身体广义速度 | 16 |
| actuator ctrl | 当前 12 维执行器控制量 | 12 |
| torso velocity | 躯干线速度（velocimeter 传感器） | 3 |
| torso upright | 躯干朝上程度 | 1 |
| imu | IMU 加速度与角速度 | 6 |

**奖励结构**（这是 Sim2Real 任务设计的精髓，值得反复读）：

$$
r_{\text{total}} = \underbrace{r_{\text{upright}} \cdot r_{\text{move}}}_{\text{任务主奖励（乘法门控）}}
+ \underbrace{r_{\text{height}} + r_{\text{lateral}} + r_{\text{heading}} + r_{\text{smooth}}}_{\text{塑形项 shaping}}
- \underbrace{(\text{后退} + \text{竖直速度} + \text{横滚俯仰角速度} + \text{偏离站姿})}_{\text{惩罚项}}
$$

三个设计要点：

1. **乘法门控**：速度奖励 $r_{\text{move}}$ 乘上直立奖励 $r_{\text{upright}}$——
   摔倒了跑得再快也没用，从奖励结构上杜绝「躺平刷分」；
2. **塑形项**引导平稳步态（高度、横向稳定、朝向一致、动作平滑），
   其中动作平滑惩罚（相邻时刻动作变化）同时是重要的 Sim2Real 项——
   抖动剧烈的策略在真实电机上根本执行不了；
3. **终止条件**：最长 20 s；观测出现 NaN 即终止；walk/run/escape 当前**没有**跌倒即终止。

命令行用法与 cartpole 完全一致，只换 `--env`：

```bash
uv run scripts/train.py --env dm-quadruped-walk
uv run tensorboard --logdir runs/dm-quadruped-walk
uv run scripts/play.py --env dm-quadruped-walk
```

> 观察一个细节：观测里包含了 `actuator ctrl`（上一步动作）。为什么需要它？
> 因为奖励里要惩罚「相邻时刻动作变化」，策略需要知道自己上一步输出了什么；
> 同时它也隐式编码了执行器的状态——这是 W16 讲动作延迟时我们还会回到的话题。

## ✏️ 练习

**E1（★，约 15 分钟）｜命令默写**
不看前文，写出在 MotrixLab 中训练 `dm-quadruped-run` 的三件套命令（预览/训练/测试），
训练要求使用 RSLRL 框架、4096 个并行环境。
**交付物**：三条命令 + 每个参数的一句话解释。

**E2（★，约 20 分钟）｜超参直觉**
在 2.4 的 SB3 代码中把 `n_steps` 从默认 2048 改为 256（并相应把 `batch_size` 设为 256）重训，对比同样 100k 步下的评估回报。
**交付物**：两组回报数字 + 2 句话解释 `n_steps` 对 PPO 样本效率/稳定性的影响
（联系 MotrixLab 默认 `--num-envs 2048` 的设计意图）。

**E3（★★，约 40 分钟）｜观测设计分析**
基于 2.5 的观测表回答：四足环境为什么不直接把「世界坐标系下的躯干位置」放进观测，
而使用本体（egocentric）坐标系下的量？把全局位置放进观测会有什么后果？（提示：泛化与 DR）
**交付物**：200–300 字分析。

**E4（★★★，约 60 分钟）｜跑通 MotrixLab**（需本机安装）
按 `docs/phase2_motrixlab.md` 完成 MotrixLab 独立安装，跑通 cartpole 三件套，
在 TensorBoard 中找到回报曲线对应的标量名称。
**交付物**：终端训练日志片段（含最终回报）+ TensorBoard 中标量名列表 + 一张曲线截图或描述。

<details>
<summary>参考答案</summary>

**E1**：

```bash
uv run scripts/view.py  --env dm-quadruped-run                    # 预览环境（随机动作）
uv run scripts/train.py --env dm-quadruped-run --rllib rslrl --num-envs 4096  # RSLRL + 4096 并行环境
uv run scripts/play.py  --env dm-quadruped-run                    # 自动加载最佳策略测试
```

`--env` 指定注册的环境名；`--rllib rslrl` 选择 PyTorch 后端的 RSLRL 框架；
`--num-envs 4096` 把并行环境数从默认 2048 提到 4096（更多样本吞吐，吃更多 CPU/内存）。

**E2**：`n_steps=2048` 时单次更新用更多数据、梯度估计更稳，但 30k 步内更新次数变少，
最终回报可能略低或方差更小（两种结果都可能，重点在解释）。MotrixLab 用 `--num-envs 2048`
并行采样，本质上是**用空间（并行环境）换时间（单环境 rollout 长度）**——每次更新照样拿到
大批量数据，但墙钟时间远小于单环境长 rollout。

**E3**：本体坐标系下的观测与「机器人站在世界何处」无关，策略学到的是可迁移的运动技能
（在任何位置都成立的反射式控制）；若把全局位置放进观测，策略可能过拟合到训练时的
初始位置附近，换个出发点就失效，也无法直接迁移到真实机器人（真机上没有全局定位的
免费真值）。这也是为什么 `escape`/`fetch` 任务只是把原点/球/目标**相对于本体**的位置
加进观测。

**E4**：参考流程——`git lfs pull` 拉取资产；`uv sync --all-packages --all-extras`；
`uv run scripts/view.py --env cartpole` 看到倒立摆窗口；`uv run scripts/train.py --env cartpole`
训练结束后 `runs/cartpole/` 下出现检查点与 event 文件；TensorBoard 中回报类标量通常含
`Reward` / `Episode reward` 字样（具体名称以 SKRL/RSLRL 版本为准）。把日志最后几行与
曲线保存到 `journal/` 即完成。

</details>

## 延伸阅读

- [MotrixLab 官方文档](https://motrixlab.readthedocs.io/) 与 [GitHub 仓库](https://github.com/Motphys/MotrixLab)
- [快速入门：Hello MotrixLab](https://motrixlab.readthedocs.io/zh-cn/latest/user_guide/getting_started/hello_motrixlab.html) —— 本讲三件套的官方出处
- [训练执行和结果分析](https://motrixlab.readthedocs.io/zh-cn/latest/user_guide/tutorial/training_and_result.html) —— 命令行参数与配置体系
- [四足机器人任务文档](https://motrixlab.readthedocs.io/zh-cn/latest/user_guide/demo/dm_quadruped.html) —— 2.5 节的完整版本，含 escape/fetch 的奖励设计
- [谋先飞 × Xbotics 实训营](https://github.com/Xbotics-Embodied-AI-club/Motphys-Xbotics-Robot-Rl-Sim-Training-Camp) —— 含 MotrixLab 入门课程与学员实战项目（如 Go2 行走），可作本阶段的伴学材料
- 训练框架：[SKRL](https://skrl.readthedocs.io/) 与 [rsl_rl](https://github.com/leggedrobotics/rsl_rl)（ETH 出品的 PPO 实现，四足社区事实标准之一）